# MILP v1 — Campus Microgrid Sizing (Gurobi)

Two-stage stochastic MILP for optimal PV, BESS, and contract demand sizing.

- **Stage 1 (here-and-now):** PV capacity, BESS power/energy, contract demand
- **Stage 2 (recourse):** Hourly dispatch across 95 representative days × 5 scenarios
- **Solver:** Gurobi (requires academic license — free for students)

Uses shared config/data from `milp_common.py`.

> **License:** Run `grbgetkey <your-key>` once to activate. See https://www.gurobi.com/academia/academic-program-and-licenses/

In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import time
from milp_common import get_config, load_data, print_results

In [ ]:
CFG = get_config()
scenarios, meta, repday_data, n_repdays, n_scenarios, n_hours = load_data(CFG)

## Build MILP Model

In [ ]:
m = gp.Model('MicrogridSizing')
m.Params.TimeLimit = CFG['time_limit']

# === Stage 1: Capacity variables ===
cap_pv = m.addVar(ub=CFG['pv_max_kw'], name='cap_pv')
cap_bess_p = m.addVar(ub=CFG['bess_p_max_kw'], name='cap_bess_p')
cap_bess_e = m.addVar(ub=CFG['bess_e_max_kwh'], name='cap_bess_e')
cap_contract = m.addVar(name='cap_contract')

# === Stage 2: Dispatch variables ===
DST = [(d, s, t) for d in range(n_repdays)
       for s in range(n_scenarios) for t in range(n_hours)]

p_grid = m.addVars(DST, name='grid')
p_curt = m.addVars(DST, name='curt')
p_export = m.addVars(DST, name='exp')
p_ch = m.addVars(DST, name='ch')
p_dis = m.addVars(DST, name='dis')
soc = m.addVars(DST, name='soc')
over_contract = m.addVars(DST, name='oc')

m.update()
print(f'Variables created: {m.NumVars:,}')

## Objective & Constraints

In [ ]:
# === Investment cost ===
inv_cost = (
    cap_pv * CFG['capex_pv_per_kw'] * CFG['crf_pv']
    + cap_bess_p * CFG['capex_bess_power_per_kw'] * CFG['crf_bess']
    + cap_bess_e * (CFG['capex_bess_energy_per_kwh'] * CFG['crf_bess']
                    + CFG['capex_bess_energy_per_kwh'] * CFG['fom_bess_rate'])
    + cap_contract * CFG['contract_price_per_kw_month'] * 12
)

op_cost = gp.LinExpr(0)
re_total = gp.LinExpr(0)
load_total_val = 0.0  # constant accumulator

PV_RATING = 50.0

for d in range(n_repdays):
    dd = repday_data[d]
    w = dd['weight']
    tou = dd['tou']

    for s in range(n_scenarios):
        sc = dd['scenarios'][s]
        pw = sc['prob'] * w
        pv_avail = sc['pv_kw']
        load_kw = sc['load_kw']

        for t in range(n_hours):
            idx = (d, s, t)
            pv_coeff = float(pv_avail[t]) / PV_RATING
            load_t = float(load_kw[t])

            # --- Power balance ---
            m.addConstr(
                pv_coeff * cap_pv + p_grid[idx] + p_dis[idx]
                == load_t + p_ch[idx] + p_curt[idx] + p_export[idx],
                name=f'bal_{d}_{s}_{t}'
            )

            # --- BESS limits ---
            m.addConstr(p_ch[idx] <= cap_bess_p, name=f'ch_lim_{d}_{s}_{t}')
            m.addConstr(p_dis[idx] <= cap_bess_p, name=f'dis_lim_{d}_{s}_{t}')

            # --- SOC dynamics ---
            if t == 0:
                soc_prev = CFG['soc_init'] * cap_bess_e
            else:
                soc_prev = soc[(d, s, t - 1)]

            m.addConstr(
                soc[idx] == soc_prev
                + CFG['eff_charge'] * p_ch[idx]
                - (1.0 / CFG['eff_discharge']) * p_dis[idx],
                name=f'soc_{d}_{s}_{t}'
            )

            m.addConstr(soc[idx] >= CFG['soc_min'] * cap_bess_e, name=f'soc_lo_{d}_{s}_{t}')
            m.addConstr(soc[idx] <= CFG['soc_max'] * cap_bess_e, name=f'soc_hi_{d}_{s}_{t}')

            # --- Over-contract penalty ---
            m.addConstr(over_contract[idx] >= p_grid[idx] - cap_contract,
                        name=f'oc_{d}_{s}_{t}')

            # --- Export limit ---
            m.addConstr(p_export[idx] <= pv_coeff * cap_pv, name=f'exp_lim_{d}_{s}_{t}')

            # --- Operating cost ---
            op_cost += pw * (
                p_grid[idx] * float(tou[t])
                + over_contract[idx] * CFG['penalty_per_kw']
                - p_export[idx] * CFG['feed_in_tariff']
                + (p_ch[idx] + p_dis[idx]) * CFG['bess_degradation_cost']
            )

            # --- RE accounting ---
            re_total += pw * (load_t - p_grid[idx])
            load_total_val += pw * load_t

m.setObjective(inv_cost + op_cost, GRB.MINIMIZE)

# === RE target constraint ===
m.addConstr(re_total >= CFG['re_target'] * load_total_val, name='RE30')

m.update()
print(f'Constraints: {m.NumConstrs:,}')
print(f'Variables:   {m.NumVars:,}')
print('Model built.')

## Solve

In [ ]:
t0 = time.time()
m.optimize()
solve_time = time.time() - t0

print(f'\nStatus: {m.Status} ({"Optimal" if m.Status == GRB.OPTIMAL else "See Gurobi docs"})')
print(f'Objective: {m.ObjVal:,.0f} TWD')
print(f'Solve time: {solve_time:.1f}s')

## Results

In [ ]:
capacities = (
    cap_pv.X,
    cap_bess_p.X,
    cap_bess_e.X,
    cap_contract.X,
)
obj_val = m.ObjVal

# Compute RE values from solution
re_val = re_total.getValue()
load_val = load_total_val

results = print_results(
    capacities, obj_val, CFG, re_val, load_val, solve_time,
    repday_data, n_repdays, n_scenarios, n_hours, scenarios
)